# Where the Price Isn't Published

### Estimating apartment prices in Finland's thin housing markets and what they reveal about affordability

---

## Project Summary

Statistics Finland publishes apartment prices in euros per square metre (€/m²) for Finnish postal code areas when enough housing transactions have occurred. In areas with very few transactions, the price is suppressed to protect the privacy of individual buyers and sellers.

This creates a systematic information gap. Many smaller towns, rural centres, and low-transaction housing markets have an existing apartment stock but no published price information. As a result, assessing housing affordability in these areas becomes difficult.

This project investigates whether neighbourhood-level socioeconomic, demographic, and housing characteristics can be used to estimate apartment prices for postal code areas where the official price is unavailable.

### Research Question

> **Can neighbourhood characteristics available in Statistics Finland's Paavo open data explain enough variation in apartment prices to produce useful estimates for postal code areas where prices are suppressed?**

### Target Variable

The target variable is:

**Average apartment price per square metre (€/m²) in a postal code area for a given year.**

### Hypothesis

Neighbourhood socioeconomic and built-environment characteristics are expected to explain a substantial share of the variation in apartment prices between Finnish postal code areas.

In particular, I expect:

- higher household income to be associated with higher apartment prices,
- a higher share of residents with higher education to be associated with higher prices,
- unemployment and household composition to contribute to differences between areas,
- dwelling characteristics and tenure structure to influence local prices,
- and municipality or regional location to explain a substantial part of price variation because Finnish housing markets are geographically segmented.

The relationship can be summarised as:

**Neighbourhood characteristics → Apartment price (€/m²)**

### Approach

The project uses annual postal-code-level data from **2020–2024**.

The modelling workflow is:

1. Combine apartment price data with Paavo neighbourhood characteristics by postal code and year.
2. Identify areas where apartment prices are published, suppressed, or unavailable because no transactions occurred.
3. Train and evaluate regression models using only observations with known prices.
4. Use a grouped validation strategy so that the same postal code does not appear in both training and test data.
5. Compare model performance using **RMSE, MAE, and R²**.
6. Select a final model based on predictive performance, interpretability, and suitability for the intended use.
7. Apply the selected model to areas where apartment prices are suppressed.
8. Use published or estimated prices together with household income to explore housing affordability.

### Housing Affordability Application

For each postal code area, a simplified affordability indicator will be calculated as:

**Estimated cost of a 50 m² apartment ÷ average household disposable income**

This price-to-income measure can be combined with indicators such as recent dwelling-stock growth to identify areas where housing costs appear high relative to local incomes and where housing supply has not expanded correspondingly.

The resulting estimates are intended for **screening, comparison, and policy-oriented analysis**, not for individual property valuation.

### Stakeholders

Potential users of the results include:

| Stakeholder | Potential use |
|---|---|
| Municipal housing planners | Identify areas where affordability pressure may justify housing or zoning interventions |
| Regional development organisations | Compare affordability and housing-supply conditions between areas |
| Statistics Finland and open-data users | Understand where suppressed housing-price statistics create information gaps |
| Researchers and residents | Obtain approximate area-level housing-market context where published prices are unavailable |

### Success Criteria

The objective is not simply to maximise R².

A successful model should:

- achieve reasonable predictive accuracy on unseen postal code areas,
- distinguish meaningfully between areas with different housing-price levels,
- remain interpretable where possible,
- perform consistently under grouped validation,
- and clearly communicate where predictions are uncertain or potentially biased.

Model performance will primarily be evaluated using:

- **RMSE** — Root Mean Squared Error
- **MAE** — Mean Absolute Error
- **R²** — Coefficient of Determination

### Important Limitation

Missing prices are **not random**.

Postal code areas with suppressed prices tend to have fewer housing transactions than the areas used to train the model. Therefore, even a carefully designed held-out test score may be somewhat optimistic when compared with the model's true performance on suppressed-price areas.

Predicted values will therefore be presented explicitly as **model estimates rather than official Statistics Finland statistics**.

### Data Sources

The project uses open aggregated data from **Statistics Finland**:

- **Paavo postal code area statistics** — socioeconomic, demographic, household, dwelling, employment, and education characteristics.
- **Prices of dwellings by postal code area** — apartment prices per square metre and transaction counts.

The analysis covers **2020–2024**.

No individual-level or personally identifying information is used. All data is pre-aggregated by Statistics Finland.

---

## Notebook Structure

1. **Title and Summary**
2. **Imports and Configuration**
3. **Data Loading and Preparation**
4. **Exploration and Visualisation**
5. **Modelling**
6. **Evaluation and Diagnostics**
7. **Conclusions, Stakeholder Summary, and Limitations**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load files

## Load target files

In [2]:
# Load 1 dataset for check its structure
df_t_test = pd.read_csv("../data/raw/price_per_square_meter_2020.csv")
df_t_test.head()

,,,,,,"13mu -- Prices per square meter of old dwellings in housing companies and numbers of transactions by postal code area, yearly, 2009-2025"
Postal code,"Price per square meter (EUR/m2) 2020 Blocks of flats, one-room flat","Price per square meter (EUR/m2) 2020 Blocks of flats, two-room flat","Price per square meter (EUR/m2) 2020 Blocks of flats, three-room flat+","Number of sales, asset transfer tax data starting from 2020 2020 Blocks of flats, one-room flat","Number of sales, asset transfer tax data starting from 2020 2020 Blocks of flats, two-room flat","Number of sales, asset transfer tax data start..."
00100 Helsinki keskusta - Etu-Töölö (Helsinki),8419,7773,7153,110,178,195
00120 Punavuori - Bulevardi (Helsinki),8702,7993,8065,55,82,75
00130 Kaartinkaupunki (Helsinki),...,8588,8083,5,11,19
00140 Kaivopuisto - Ullanlinna (Helsinki),8948,8416,8811,73,69,92


In [3]:
def load_year(path, year):
    df = pd.read_csv(path, skiprows=2, encoding="utf-8-sig")

    df.columns = [
        "postal_raw",
        "price_1", "price_2", "price_3",
        "sales_1", "sales_2", "sales_3"
    ]

    rows = []

    for rooms in [1, 2, 3]:
        temp = df[["postal_raw", f"price_{rooms}", f"sales_{rooms}"]].copy()
        temp.columns = ["postal_raw", "price_per_sqm", "n_sales"]
        temp["rooms"] = rooms
        rows.append(temp)

    out = pd.concat(rows, ignore_index=True)
    out["year"] = year

    return out

In [4]:
years = [2020, 2021, 2022, 2023, 2024]

df = pd.concat(
    [load_year(f"../data/raw/price_per_square_meter_{y}.csv", y) for y in years],
    ignore_index=True,
)

In [5]:
df["postal_code"]  = df["postal_raw"].str[:5]
df["municipality"] = df["postal_raw"].str.extract(r"\((.*?)\)$")

df["price_per_sqm"] = pd.to_numeric(df["price_per_sqm"], errors="coerce")
df["n_sales"]       = pd.to_numeric(df["n_sales"], errors="coerce")

df = df[["postal_code", "municipality", "year", "rooms", "price_per_sqm", "n_sales"]]

In [6]:
display(df.shape)
df.head()

(25860, 6)

,postal_code,municipality,year,rooms,price_per_sqm,n_sales
0,00100,Helsinki,2020,1,8419.0,110.0
1,00120,Helsinki,2020,1,8702.0,55.0
2,00130,Helsinki,2020,1,NaN,5.0
3,00140,Helsinki,2020,1,8948.0,73.0
4,00150,Helsinki,2020,1,8739.0,134.0


### Prepare price dataset for merge

In [7]:
def create_area_years_prices (data):

  data = data.copy()

  data['weighted_price'] = data['price_per_sqm'] * data['n_sales']

  area_year = data.groupby(['postal_code', 'year'], as_index=False).agg(
    municipality=("municipality", "first"),
    total_sales = ('n_sales', 'sum'),
    weighted_sum = ('weighted_price', 'sum')
  )

  area_year['average_price'] = area_year['weighted_sum'] / area_year['total_sales']

  return area_year.drop(columns='weighted_sum')

In [8]:
df_year_prices = create_area_years_prices(df)
display(df_year_prices.shape)
df_year_prices.head()

(8620, 5)

,postal_code,year,municipality,total_sales,average_price
0,00100,2020,Helsinki,483.0,7669.811594
1,00100,2021,Helsinki,669.0,8309.665172
2,00100,2022,Helsinki,475.0,8398.305263
3,00100,2023,Helsinki,329.0,7753.027356
4,00100,2024,Helsinki,335.0,7402.459701


In [9]:
df_year_prices["status"] = "published"
df_year_prices.loc[df_year_prices.average_price.isna() & df_year_prices.total_sales.notna(), "status"] = "suppressed"
df_year_prices.loc[df_year_prices.total_sales.isna(), "status"] = "no_sales"

df_year_prices.head()

,postal_code,year,municipality,total_sales,average_price,status
0,00100,2020,Helsinki,483.0,7669.811594,published
1,00100,2021,Helsinki,669.0,8309.665172,published
2,00100,2022,Helsinki,475.0,8398.305263,published
3,00100,2023,Helsinki,329.0,7753.027356,published
4,00100,2024,Helsinki,335.0,7402.459701,published


## Load features files

In [10]:
# Load 1 dataset to check its structure
df_f_test = pd.read_csv('../data/raw/average_floor_area_per_dwelling_2020_2024.csv')
df_f_test.head()

,,,,,"12f4 -- 6. Buildings and dwellings, 2010-2024"
Postal code area,2020 Average floor area per dwelling (RA),2021 Average floor area per dwelling (RA),2022 Average floor area per dwelling (RA),2023 Average floor area per dwelling (RA),2024 Average floor area per dwelling (RA)
00100 Helsinki keskusta - Etu-Töölö (Helsinki),65.8,65.8,65.7,65.6,65.7
00120 Punavuori - Bulevardi (Helsinki),69.5,69.6,69.8,70.0,70.0
00130 Kaartinkaupunki (Helsinki),78.5,78.4,79.3,80.2,81.0
00140 Kaivopuisto - Ullanlinna (Helsinki),74.1,74.1,74.4,74.2,74.4


In [11]:
paavo_files = {
    "population":        "../data/raw/population_structure_2020_2024.csv",
    "age18plus":         "../data/raw/age_18_or_over_2020_2024.csv",
    "higher_degree":     "../data/raw/academic_degree_higher_university_level_degree_2020_2024.csv",
    "income":            "../data/raw/average_income_of_households_2020_2024.csv",
    "hh_size":           "../data/raw/average_size_of_households_2020_2024.csv",
    "hh_total":          "../data/raw/households_total_2020_2024.csv",
    "hh_rented":         "../data/raw/households_living_in_rented_dwellings_2020_2024.csv",
    "hh_one_person":     "../data/raw/one_person_households_2020_2024.csv",
    "dwellings":         "../data/raw/dwellings_2020_2024.csv",
    "dwellings_flats":   "../data/raw/dwellings_in_blocks_of_flats_2020_2024.csv",
    "avg_floor_area":    "../data/raw/average_floor_area_per_dwelling_2020_2024.csv",
    "unemployed":        "../data/raw/unemployment_new_2020_2024.csv",
    "workplaces":        "../data/raw/workplaces_total_2020_2024.csv",
}

In [12]:
def load_paavo(path, name):
    df = pd.read_csv(path, skiprows=2, encoding="utf-8-sig")

    df = df.rename(columns={df.columns[0]: "postal_raw"})
    df = df.melt(id_vars="postal_raw", var_name="year", value_name=name)

    df["year"] = df["year"].str.extract(r"(20\d{2})").astype(int)
    df["postal_code"] = df["postal_raw"].str[:5]
    df[name] = pd.to_numeric(df[name], errors="coerce")

    return df[["postal_code", "year", name]]

In [13]:
frames = [load_paavo(p, n) for n, p in paavo_files.items()]

paavo = frames[0]

for frame in frames[1:]:
    paavo = paavo.merge(
        frame,
        on=["postal_code", "year"],
        how="outer"
    )

paavo.loc[paavo["year"] == 2024, "workplaces"] = pd.NA

In [14]:
print(paavo.shape)        # expect ~15,000 rows (3018 × 5 years)
paavo.head()

(15090, 15)


,postal_code,year,population,age18plus,higher_degree,income,hh_size,hh_total,hh_rented,hh_one_person,dwellings,dwellings_flats,avg_floor_area,unemployed,workplaces
0,00100,2020,18373,16236,6023.0,70284.0,1.7,10380,5289.0,5379.0,12341.0,11721.0,65.8,1083.0,49712.0
1,00100,2021,17893,15817,5949.0,73553.0,1.7,10141,5157.0,5267.0,12321.0,11663.0,65.8,723.0,52491.0
2,00100,2022,18030,15997,5977.0,70016.0,1.7,10280,5334.0,5447.0,12337.0,11719.0,65.7,630.0,55404.0
3,00100,2023,18462,16365,6225.0,70761.0,1.7,10568,5527.0,5620.0,12440.0,11819.0,65.6,693.0,55646.0
4,00100,2024,18492,16397,6422.0,73915.0,1.7,10544,5352.0,5546.0,12469.0,11869.0,65.7,834.0,NaN


## Load geographic features

In [15]:
import geopandas as gpd

# Load postal-code geometries
gdf = gpd.read_file(
    "https://geo.stat.fi/geoserver/postialue/wfs"
    "?service=WFS&version=2.0.0&request=GetFeature"
    "&typeName=postialue:pno_2024&outputFormat=json"
)

# Rename columns into English
gdf = gdf.rename(columns={
    "posti_alue": "postal_code",
    "vuosi": "year",
    "nimi": "name_fi",
    "namn": "name_sv",
    "kunta": "municipality_code",
    "kuntanro": "municipality_number",
    "pinta_ala": "area_m2"
})

gdf.head()

,id,objectid,postal_code,year,name_fi,name_sv,municipality_code,municipality_number,area_m2,geometry
0,1,1,00100,2024,Helsinki keskusta - Etu-Töölö,Helsingfors centrum - Främre Tölö,091,91,2353278,"POLYGON ((385653.893 6671591.048, 385779.642 6..."
1,2,2,00120,2024,Punavuori - Bulevardi,Rödbergen - Bulevarden,091,91,414010,"POLYGON ((385316.092 6671076.984, 385388.492 6..."
2,3,3,00130,2024,Kaartinkaupunki,Gardesstaden,091,91,428960,"POLYGON ((386212.111 6671061.262, 386298.022 6..."
3,5,5,00150,2024,Punavuori - Eira - Hernesaari,Rödbergen - Eira - Ärtholmen,091,91,1367328,"MULTIPOLYGON (((384846.102 6669565.816, 384867..."
4,32,32,00420,2024,Kannelmäki,Gamlas,091,91,2890057,"POLYGON ((382632.09 6682138.433, 382483.046 66..."


In [16]:
# Project to EPSG:3067
# Coordinates and distances are now in metres
gdf = gdf.to_crs(epsg=3067)

# Postal-code centroid coordinates
gdf["centroid"] = gdf.geometry.centroid

gdf["centroid_x"] = gdf["centroid"].x
gdf["centroid_y"] = gdf["centroid"].y

In [17]:
gdf.head()

,id,objectid,postal_code,year,name_fi,name_sv,municipality_code,municipality_number,area_m2,geometry,centroid,centroid_x,centroid_y
0,1,1,00100,2024,Helsinki keskusta - Etu-Töölö,Helsingfors centrum - Främre Tölö,091,91,2353278,"POLYGON ((385653.893 6671591.048, 385779.642 6...",POINT (385114.002 6672390.769),385114.002032,6.672391e+06
1,2,2,00120,2024,Punavuori - Bulevardi,Rödbergen - Bulevarden,091,91,414010,"POLYGON ((385316.092 6671076.984, 385388.492 6...",POINT (385613.52 6671377.831),385613.519532,6.671378e+06
2,3,3,00130,2024,Kaartinkaupunki,Gardesstaden,091,91,428960,"POLYGON ((386212.111 6671061.262, 386298.022 6...",POINT (386227.615 6671492.049),386227.614544,6.671492e+06
3,5,5,00150,2024,Punavuori - Eira - Hernesaari,Rödbergen - Eira - Ärtholmen,091,91,1367328,"MULTIPOLYGON (((384846.102 6669565.816, 384867...",POINT (385117.961 6670308.479),385117.961363,6.670308e+06
4,32,32,00420,2024,Kannelmäki,Gamlas,091,91,2890057,"POLYGON ((382632.09 6682138.433, 382483.046 66...",POINT (382610.119 6680386.839),382610.118643,6.680387e+06


## Merge All Datasets

In [18]:
# Merge paavo dataset with geo_features
geo_features = gdf[
    [
        "postal_code",
        "municipality_code",
        "centroid_x",
        "centroid_y",
        "area_m2"
    ]
].copy()

paavo = paavo.merge(
    geo_features,
    on="postal_code",
    how="left"
)

In [19]:
paavo.head()

,postal_code,year,population,age18plus,higher_degree,income,hh_size,hh_total,hh_rented,hh_one_person,dwellings,dwellings_flats,avg_floor_area,unemployed,workplaces,municipality_code,centroid_x,centroid_y,area_m2
0,00100,2020,18373,16236,6023.0,70284.0,1.7,10380,5289.0,5379.0,12341.0,11721.0,65.8,1083.0,49712.0,091,385114.002032,6.672391e+06,2353278
1,00100,2021,17893,15817,5949.0,73553.0,1.7,10141,5157.0,5267.0,12321.0,11663.0,65.8,723.0,52491.0,091,385114.002032,6.672391e+06,2353278
2,00100,2022,18030,15997,5977.0,70016.0,1.7,10280,5334.0,5447.0,12337.0,11719.0,65.7,630.0,55404.0,091,385114.002032,6.672391e+06,2353278
3,00100,2023,18462,16365,6225.0,70761.0,1.7,10568,5527.0,5620.0,12440.0,11819.0,65.6,693.0,55646.0,091,385114.002032,6.672391e+06,2353278
4,00100,2024,18492,16397,6422.0,73915.0,1.7,10544,5352.0,5546.0,12469.0,11869.0,65.7,834.0,NaN,091,385114.002032,6.672391e+06,2353278


In [20]:
df_merged = df_year_prices.merge(
  paavo,
  on=['postal_code', 'year'],
  how='left'
)

display(df_merged.shape)
df_merged.head()

(8620, 23)

,postal_code,year,municipality,total_sales,average_price,status,population,age18plus,higher_degree,income,...,hh_one_person,dwellings,dwellings_flats,avg_floor_area,unemployed,workplaces,municipality_code,centroid_x,centroid_y,area_m2
0,00100,2020,Helsinki,483.0,7669.811594,published,18373.0,16236.0,6023.0,70284.0,...,5379.0,12341.0,11721.0,65.8,1083.0,49712.0,091,385114.002032,6.672391e+06,2353278.0
1,00100,2021,Helsinki,669.0,8309.665172,published,17893.0,15817.0,5949.0,73553.0,...,5267.0,12321.0,11663.0,65.8,723.0,52491.0,091,385114.002032,6.672391e+06,2353278.0
2,00100,2022,Helsinki,475.0,8398.305263,published,18030.0,15997.0,5977.0,70016.0,...,5447.0,12337.0,11719.0,65.7,630.0,55404.0,091,385114.002032,6.672391e+06,2353278.0
3,00100,2023,Helsinki,329.0,7753.027356,published,18462.0,16365.0,6225.0,70761.0,...,5620.0,12440.0,11819.0,65.6,693.0,55646.0,091,385114.002032,6.672391e+06,2353278.0
4,00100,2024,Helsinki,335.0,7402.459701,published,18492.0,16397.0,6422.0,73915.0,...,5546.0,12469.0,11869.0,65.7,834.0,NaN,091,385114.002032,6.672391e+06,2353278.0


In [21]:
display(df_merged.shape)
df_merged.columns

(8620, 23)

Index(['postal_code', 'year', 'municipality', 'total_sales', 'average_price',
       'status', 'population', 'age18plus', 'higher_degree', 'income',
       'hh_size', 'hh_total', 'hh_rented', 'hh_one_person', 'dwellings',
       'dwellings_flats', 'avg_floor_area', 'unemployed', 'workplaces',
       'municipality_code', 'centroid_x', 'centroid_y', 'area_m2'],
      dtype='str')

# Assess Data Quality (Initial EDA)

In [22]:
df_merged.describe()

,year,total_sales,average_price,population,age18plus,higher_degree,income,hh_size,hh_total,hh_rented,hh_one_person,dwellings,dwellings_flats,avg_floor_area,unemployed,workplaces,centroid_x,centroid_y,area_m2
count,8620.000000,8620.000000,4567.000000,8615.000000,8615.000000,8597.000000,8566.000000,8566.000000,8615.000000,8566.000000,8566.000000,8610.000000,8610.000000,8610.000000,8602.000000,6892.000000,8615.000000,8.615000e+03,8.615000e+03
mean,2022.000000,26.553944,1214.938662,3022.947649,2465.840279,320.120972,46138.563390,2.043159,1554.899362,605.454821,731.769904,1752.956446,879.481998,89.826702,162.100791,1229.389437,398205.417708,6.880563e+06,1.227434e+08
std,1.414296,62.523662,1641.310788,3648.253255,3032.068841,578.126977,12220.416119,0.329438,2015.084946,1131.246177,1117.852108,2233.402316,1880.289539,17.642843,234.396208,2814.214184,115270.215910,1.993208e+05,3.341196e+08
min,2020.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,186133.272160,6.639874e+06,1.763510e+05
25%,2021.000000,0.000000,0.000000,537.000000,441.000000,26.000000,38619.000000,1.800000,253.000000,24.000000,84.000000,311.000000,3.000000,79.500000,23.000000,99.000000,312891.199587,6.716958e+06,8.274848e+06
50%,2022.000000,1.000000,599.894737,1658.000000,1333.000000,86.000000,43812.500000,2.000000,781.000000,148.000000,281.000000,913.000000,70.000000,90.650000,70.000000,369.000000,387003.121984,6.823992e+06,4.205106e+07
75%,2023.000000,21.000000,1948.226549,4222.000000,3432.500000,343.000000,51057.750000,2.200000,2122.000000,662.000000,933.750000,2402.000000,805.500000,101.900000,211.000000,1239.000000,475674.385608,6.978167e+06,1.144199e+08
max,2024.000000,810.000000,9651.568306,31156.000000,28743.000000,6422.000000,201375.000000,4.000000,21145.000000,13367.000000,13739.000000,24952.000000,22365.000000,179.600000,2816.000000,55646.000000,706290.279087,7.755327e+06,7.052405e+09


### Check duplicates

In [23]:
# Check exact duplicates
df_merged.duplicated().sum()

np.int64(0)

In [25]:
# Check subset duplicates
df_merged.duplicated(subset=['postal_code', 'year']).sum()

np.int64(0)

### Check missing values

In [26]:
missing = pd.DataFrame({
  'missing_count': df_merged.isnull().sum(),
  'missing_percentage': df_merged.isnull().mean()*100
})

missing.sort_values('missing_percentage', ascending=False).round(2)

,missing_count,missing_percentage
average_price,4053,47.02
workplaces,1728,20.05
income,54,0.63
hh_size,54,0.63
hh_rented,54,0.63
hh_one_person,54,0.63
higher_degree,23,0.27
unemployed,18,0.21
dwellings_flats,10,0.12
avg_floor_area,10,0.12


### Check outliers

In [27]:
df_merged.columns

Index(['postal_code', 'year', 'municipality', 'total_sales', 'average_price',
       'status', 'population', 'age18plus', 'higher_degree', 'income',
       'hh_size', 'hh_total', 'hh_rented', 'hh_one_person', 'dwellings',
       'dwellings_flats', 'avg_floor_area', 'unemployed', 'workplaces',
       'municipality_code', 'centroid_x', 'centroid_y', 'area_m2'],
      dtype='str')

In [29]:
num_cols = [
  'average_price', 'total_sales', 'workplaces', 'income', 'hh_size', 'hh_rented', 'hh_one_person', 
  'higher_degree', 'unemployed', 'avg_floor_area', 'dwellings', 'dwellings_flats', 'hh_total', 'area_m2', 'age18plus',
  'population'
]

outlier_dict = {}

for num_col in num_cols:
  Q1 = df_merged[num_col].quantile(0.25)
  Q3 = df_merged[num_col].quantile(0.75)
  IQR = Q3 - Q1

  lower_bound = Q1 - (1.5*IQR)
  upper_bound = Q3 + (1.5*IQR)

  outliers = (df_merged[num_col]<lower_bound) | (df_merged[num_col]>upper_bound)
  outlier_dict[num_col] = outliers.tolist()

  print(f"{num_col}: {outliers.sum()} outliers.")


average_price: 195 outliers.
total_sales: 1300 outliers.
workplaces: 680 outliers.
income: 308 outliers.
hh_size: 108 outliers.
hh_rented: 944 outliers.
hh_one_person: 717 outliers.
higher_degree: 989 outliers.
unemployed: 675 outliers.
avg_floor_area: 71 outliers.
dwellings: 545 outliers.
dwellings_flats: 1183 outliers.
hh_total: 581 outliers.
area_m2: 795 outliers.
age18plus: 552 outliers.
population: 530 outliers.


### Zero values

In [30]:
zero_summary = pd.DataFrame({
  '0_count': (df_merged[num_cols] == 0).sum(),
  '0_percentage': (df_merged[num_cols]==0).mean()*100
})

zero_summary.sort_values('0_percentage', ascending=False)

,0_count,0_percentage
total_sales,4053,47.018561
average_price,1996,23.155452
dwellings_flats,1844,21.392111
higher_degree,39,0.452436
hh_rented,31,0.359629
unemployed,20,0.232019
income,10,0.116009
hh_size,10,0.116009
hh_one_person,10,0.116009
hh_total,10,0.116009
